# PodForge — VoxCPM2 TTS Server

This notebook starts a VoxCPM2 server on Google Colab's free T4 GPU.

**Steps:**
1. Run all cells (Runtime → Run all)
2. Copy the ngrok public URL printed below
3. Set it as `TTS_BASE_URL` in PodForge backend

Requirements: Colab with T4 GPU (Runtime → Change runtime type → T4 GPU)

In [ ]:
# Install dependencies
!pip install -q voxcpm fastapi uvicorn pyngrok soundfile numpy

In [ ]:
# Configure ngrok (required for public URL tunneling)
from pyngrok import ngrok
ngrok.set_auth_token("3EOr384hN9MABHuFYtoCMY6Ujb5_6UDb37kBKe1Mx6E5DHUr8")
print("ngrok configured ✓")

In [ ]:
# Download and load model (~4GB, first run only)
from voxcpm import VoxCPM

model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
print(f"Model loaded! Sample rate: {model.tts_model.sample_rate}")

In [ ]:
# Start TTS server with ngrok tunnel
import io
import base64
import threading
import numpy as np
import soundfile as sf
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn

# Auto-load model if not already loaded
try:
    model
    print("Model already loaded ✓")
except NameError:
    print("Loading model...")
    from voxcpm import VoxCPM
    model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
    print(f"Model loaded! Sample rate: {model.tts_model.sample_rate}")

app = FastAPI(title="PodForge TTS Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class TTSRequest(BaseModel):
    text: str
    cfg_value: float = 2.0
    inference_timesteps: int = 10
    reference_wav_path: str | None = None

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/generate")
async def generate(req: TTSRequest):
    kwargs = {
        "text": req.text,
        "cfg_value": req.cfg_value,
        "inference_timesteps": req.inference_timesteps,
    }
    if req.reference_wav_path:
        kwargs["reference_wav_path"] = req.reference_wav_path

    wav = model.generate(**kwargs)
    buf = io.BytesIO()
    sf.write(buf, wav, model.tts_model.sample_rate, format="WAV")
    buf.seek(0)
    audio_b64 = base64.b64encode(buf.read()).decode()

    return {
        "audio_base64": audio_b64,
        "sample_rate": model.tts_model.sample_rate,
        "duration_seconds": len(wav) / model.tts_model.sample_rate,
    }

# Kill any existing process on port 8809
import subprocess
subprocess.run(["fuser", "-k", "8809/tcp"], capture_output=True)
import time; time.sleep(1)

# Start ngrok tunnel
public_url = ngrok.connect(8809)
print(f"\n{'='*60}")
print(f"PUBLIC URL: {public_url}")
print(f"Copy this URL and set it as TTS_BASE_URL in PodForge!")
print(f"{'='*60}\n")

# Run server
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8809)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("TTS server is running! Keep this cell running.")
print("Press Ctrl+C or stop the cell to shut down.")

# Keep alive
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\nShutting down...")
    ngrok.kill()

In [ ]:
# Start TTS server with ngrok tunnel
import io
import base64
import threading
import numpy as np
import soundfile as sf
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn

app = FastAPI(title="PodForge TTS Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class TTSRequest(BaseModel):
    text: str
    cfg_value: float = 2.0
    inference_timesteps: int = 10
    reference_wav_path: str | None = None

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/generate")
async def generate(req: TTSRequest):
    kwargs = {
        "text": req.text,
        "cfg_value": req.cfg_value,
        "inference_timesteps": req.inference_timesteps,
    }
    if req.reference_wav_path:
        kwargs["reference_wav_path"] = req.reference_wav_path

    wav = model.generate(**kwargs)
    buf = io.BytesIO()
    sf.write(buf, wav, model.tts_model.sample_rate, format="WAV")
    buf.seek(0)
    audio_b64 = base64.b64encode(buf.read()).decode()

    return {
        "audio_base64": audio_b64,
        "sample_rate": model.tts_model.sample_rate,
        "duration_seconds": len(wav) / model.tts_model.sample_rate,
    }

# Start ngrok tunnel
public_url = ngrok.connect(8809)
print(f"\n{'='*60}")
print(f"PUBLIC URL: {public_url}")
print(f"Copy this URL and set it as TTS_BASE_URL in PodForge!")
print(f"{'='*60}\n")

# Run server
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8809)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("TTS server is running! Keep this cell running.")
print("Press Ctrl+C or stop the cell to shut down.")

# Keep alive
try:
    while True:
        import time
        time.sleep(60)
except KeyboardInterrupt:
    print("\nShutting down...")
    ngrok.kill()